In [13]:
!pip install pandas
!pip install vaderSentiment

In [14]:
import pandas as pd
import re
import pickle
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [15]:
# Load vectorizer
vectorizer = pickle.load(open("models/vectorizer.sav", "rb"))

# Load model
nb_model = pickle.load(open("models/trad_model.sav", "rb"))
lr_model = pickle.load(open("models/lr_model.sav", "rb"))

In [16]:
def handle_negations(s: str):
    negation_pattern = r'\b(not|no|never|none|cannot|cant|wont|dont)\b[\w\s]+'
    return re.sub(negation_pattern, lambda match: match.group(0).replace(' ', '_'), s)

""" Removes unnecessary symbols from the text """
def clean_text(s: str):
    # Only retain alphanumeric and whitespace characters
    s = re.sub(pattern=rf"[^a-zA-Z0-9\s]", repl="", string=s, flags=re.IGNORECASE)

    # Convert to lowercase
    s = s.lower()

    # Remove extra whitespaces
    s = re.sub(pattern=r"\s+", repl=" ", string=s).strip()

    return s

""" Implements pipeline of pre-processing techniques """
def preprocess(text: str):
    return handle_negations(clean_text(text))

""" Sentiment Scorer """
def sentiment_scores(sentence):
    sid_obj = SentimentIntensityAnalyzer()
    sentiment_dict = sid_obj.polarity_scores(sentence)

    return sentiment_dict['compound']

In [17]:
""" Return predictions and probability """
def predict(x, sentiment_scores):
    print(np.array(x.toarray()).shape)
    # Predict on the two models
    mnb_pred = nb_model.predict_proba(x)
    lr_pred = lr_model.predict_proba(sentiment_scores.to_numpy().reshape(-1,1))

    # Model weights
    mnb_w, lr_w = 0.4, 0.6

    tot_pred = mnb_pred * mnb_w + lr_pred * lr_w

    # Get the index of the highest probability
    # return np.argmax(mnb_pred, axis=1), [max(prob) for prob in mnb_pred]
    return np.argmax(tot_pred, axis=1), [max(prob) for prob in tot_pred]

In [ ]:
# Load test data
df_test = pd.read_csv("data/test.csv")

# Preprocess data
df_test["cleaned"] = df_test["text"].apply(preprocess)
df_test["sentiment_score"] = df_test["text"].apply(sentiment_scores)

# Tokenize the dataset
X_test = vectorizer.transform(df_test["cleaned"])

# Predict the labels
result, _ = predict(X_test, df_test["sentiment_score"])

# Create a new dataframe
df_pred = pd.DataFrame({
    'text' : df_test["text"].tolist(),
    'actual_label' : df_test["label"].tolist(),
    'predicted_label' : result,
})

# Save the predictions to a CSV file
df_pred.to_csv("predictions/trad_predictions.csv", index=False)

(160, 1600)
